In [2]:
#Import libraries and define Symbols
import sympy as smp
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, RK45, BDF
from scipy.optimize import direct, Bounds
import time
from control import lqr
import scipy as sp
import pandas as pd

#Symbols
x, y, z, dx, dy, dz, phi, theta, psi, m, gx, gy, T, P, alpha, S, Cl, Cd, Ixx, Iyy, Izz, dIxx, dIyy, dIzz, dphi, dtheta, dpsi, f, sin_alpha = smp.symbols(
'x y z \\dot{x} \\dot{y} \\dot{z} phi theta psi m gamma_x gamma_y T P alpha S C_L C_D I_{xx} I_{yy} I_{zz} \\dot{I_{xx}}, \\dot{I_{yy}}, \\dot{I_{zz}} \\dot{\\phi}, \\dot{\\theta}, \\dot{\\psi} f \\sin_{\\alpha}', cls=smp.Function)
t, g, rho, dCoP, dNoz, Tau0 = smp.symbols('t g rho d_{CoP} d_{Noz} tau')

# Forces, Moments, and State

The rocket’s position and velocity are expressed in the Earth frame. Moments and angular velocity are expressed in the body frame. The unit vector $\hat{\mathbf z}$ points along the rocket’s body axis.

## Orientation and forces

Roll $\phi$, pitch $\theta$, and yaw $\psi$ define the body-to-Earth rotation. Its transpose converts Earth-frame forces to the body frame:

\begin{equation}
R_{be}=R_z(\psi)R_y(\theta)R_x(\phi),
\qquad
R_{eb}=R_{be}^{T}.
\end{equation}

The velocity and angle between the velocity and rocket axis are

\begin{equation}
\mathbf v=
\begin{bmatrix}\dot x\ \dot y\ \dot z\end{bmatrix},
\qquad
\alpha=\cos^{-1}\left(
\frac{\mathbf v\cdot R_{be}\hat{\mathbf z}}{|\mathbf v|}
\right).
\end{equation}

Gravity acts downward, thrust acts along the gimballed nozzle direction, drag opposes velocity, and lift points toward the component of the rocket axis perpendicular to velocity:

\begin{equation}
\begin{aligned}
\mathbf F_g
&=-mg\hat{\mathbf z},\\
\mathbf F_t
&=T(1-P)R_{be}R_y(g_y)R_x(g_x)\hat{\mathbf z},\\
\mathbf F_d
&=-\frac{1}{2}\rho|\mathbf v|^2S(\alpha)C_d(\alpha)
\frac{\mathbf v}{|\mathbf v|},\\
\mathbf F_l
&=\frac{1}{2}\rho|\mathbf v|^2S(\alpha)C_l(\alpha)
\frac{\bigl(\mathbf v\times R_{be}\hat{\mathbf z}\bigr)\times\mathbf v}
{|\mathbf v|^2}.
\end{aligned}
\end{equation}

The vector acceleration equation, with all four forces substituted, is

\begin{equation}
\boxed{
\mathbf{\ddot r}
=
\mathbf a
=-g\hat{\mathbf z}
+\frac{T(1-P)}{m}R_{be}R_y(g_y)R_x(g_x)\hat{\mathbf z}
+\frac{\rho|\mathbf v|^2S(\alpha)}{2m}
\left[
C_l(\alpha)
\frac{\bigl(\mathbf v\times R_{be}\hat{\mathbf z}\bigr)\times\mathbf v}
{|\mathbf v|^2}+

C_d(\alpha)\frac{\mathbf v}{|\mathbf v|}
\right].
}
\end{equation}

## Moments and angular acceleration

The nozzle and center of pressure (COP) are located at $-d_{\mathrm{Noz}}\hat{\mathbf z}$ and $-d_{\mathrm{CoP}}\hat{\mathbf z}$ relative to the center of mass (COM). Their offsets are the moment arms for thrust and aerodynamic forces, respectively. Gravity acts at the COM in this model and therefore creates no moment about it. The total body-frame moment is

\begin{equation}
\mathbf M=
(-d_{\mathrm{Noz}}\hat{\mathbf z})\times(R_{eb}\mathbf F_t)
+
(-d_{\mathrm{CoP}}\hat{\mathbf z})\times
R_{eb}(\mathbf F_l+\mathbf F_d)
+
\tau_0f(t)\hat{\mathbf z}.
\end{equation}

Let $\mathbf q=(\phi,\theta,\psi)^T$. The matrix $J$ converts body angular velocity to Euler angle rates, while $I$ is the body-frame inertia matrix:

\begin{equation}
\dot{\mathbf q}=J\boldsymbol\omega,
\qquad
\boldsymbol\omega=J^{-1}\dot{\mathbf q},
\qquad
I=\operatorname{diag}(I_{xx},I_{yy},I_{zz}).
\end{equation}

The code’s Euler angle acceleration equation, with the moment terms substituted, is

\begin{equation}
\boxed{
\begin{aligned}
\ddot{\mathbf q}
={}&JI^{-1}\Bigl[
(-d_{\mathrm{Noz}}\hat{\mathbf z})\times(R_{eb}\mathbf F_t)\
+
(-d_{\mathrm{CoP}}\hat{\mathbf z})\times
R_{eb}(\mathbf F_l+\mathbf F_d)
+\tau_0f(t)\hat{\mathbf z}\
-\dot I\boldsymbol\omega
-\boldsymbol\omega\times(I\boldsymbol\omega)
\Bigr]
+\dot J\boldsymbol\omega .
\end{aligned}
}
\end{equation}

Here $\dot I\boldsymbol\omega$ accounts for changing inertia, and $\boldsymbol\omega\times(I\boldsymbol\omega)$ accounts for the coupling between rotation axes. The term $\dot J\boldsymbol\omega$ arises because the Euler angle conversion matrix changes as the rocket rotates.

## State equation and implementation notes

For the state $\mathbf X=(\mathbf r,\mathbf{\dot r},\mathbf q,\dot{\mathbf q})^T$, the assembled state derivative is

\begin{equation}
\dot{\mathbf X}
=
\begin{bmatrix}
\mathbf{\dot r}\
\mathbf{\ddot r}\
\dot{\mathbf q}\
\ddot{\mathbf q}
\end{bmatrix}.
\end{equation}

In [ ]:
#Forces, Moments, and State

#Rotation Matrices
def Rx(phi):
    return  smp.Matrix([[1,             0,              0],
                        [0, smp.cos(phi), -smp.sin(phi)],
                        [0, smp.sin(phi),  smp.cos(phi)]])
def Ry(theta):
    return  smp.Matrix([[ smp.cos(theta), 0,  smp.sin(theta)],
                        [             0, 1,              0],
                        [-smp.sin(theta), 0,  smp.cos(theta)]])
def Rz(psi):
    return  smp.Matrix([[smp.cos(psi), -smp.sin(psi), 0],
                        [smp.sin(psi),  smp.cos(psi), 0],
                        [0,             0,              1]])
Rbe = Rz(psi(t))*Ry(theta(t))*Rx(phi(t))
Reb=Rbe.T



#Forces
v = smp.Matrix([dx(t), dy(t), dz(t)])
#Gravity
zHat = smp.Matrix([0,0,1])
Fg = -m(t)*g*zHat

#Thrust
FtHat = Rbe*Ry(gy(t))*Rx(gx(t))*zHat
Ft = FtHat*T(t)*(1-P(t))

#Lift 
v = smp.Matrix([dx(t), dy(t), dz(t)])
# Fl=smp.zeros(3,1)
#Actual Expression
LHat = (v.cross(Rbe*zHat)).cross(v)/ (v.dot(v))/sin_alpha(t) #CHANGE
L = smp.Rational(1/2)*rho*v.dot(v)*S(alpha(t))*Cl(alpha(t))
Fl = LHat * L

#Drag
D = -smp.Rational(1/2)*rho*v.dot(v)*S(alpha(t))*Cd(alpha(t))
DHat = v / smp.sqrt(v.dot(v))
Fd = DHat*D

# alpha_exp = smp.acos(v.dot(Rbe*zHat) / smp.sqrt(v.dot(v)))

#Total force
F = Fg + Ft + Fl + Fd

#Acceleration
a = F / m(t)
a.simplify()

#Moment equations

#Body to Euler rate matrix
J = smp.Matrix([[1, smp.sin(phi(t))*smp.tan(theta(t)), smp.cos(phi(t))*smp.tan(theta(t))],
                [0,smp.cos(phi(t)), -smp.sin(phi(t))],
                [0, smp.sin(phi(t)) / smp.cos(theta(t)), smp.cos(phi(t)) / smp.cos(theta(t))]])


#Angular rates
av = smp.Matrix([dphi(t), dtheta(t), dpsi(t)])
omega = J.inv()* av
omega.simplify()

#Inertia matrix and rate
I = smp.diag(Ixx(t),Iyy(t),Izz(t))
dI = smp.diag(dIxx(t),dIyy(t),dIzz(t))

#Total moment (body)
M = (-dNoz*zHat).cross(Reb*Ft) + (-dCoP*zHat).cross(Reb*(Fl+Fd)) + (Tau0) * f(t) * zHat #CHANGE
M.simplify()

#Angular acceleration (aa = [ddphi, ddtheta, ddpsi])
aa = J * I.inv() * (M - dI * omega - omega.cross(I * omega)) + J.diff(t).subs([(phi(t).diff(t),dphi(t)), (theta(t).diff(t),dtheta(t)), (psi(t).diff(t),dpsi(t))]) * omega
aa.simplify()


#Total state Equation
State = smp.Matrix([v, a, av, aa])


#Sub in simpler expressions for functions and variables
Statesub = State.subs([(dx(t), smp.Symbol('dx')), (dy(t), smp.Symbol('dy')), (dz(t), smp.Symbol('dz')), 
                    (phi(t), smp.Symbol('phi')), (theta(t), smp.Symbol('theta')), (psi(t), smp.Symbol('psi')), 
                    (dphi(t), smp.Symbol('dphi')), (dtheta(t), smp.Symbol('dtheta')), (dpsi(t), smp.Symbol('dpsi')), 
                    (gx(t), smp.Symbol('gx')), (gy(t), smp.Symbol('gy')), (P(t), smp.Symbol('P')), 
                    (Cd(alpha(t)), smp.Symbol('Cd0')), (Cl(alpha(t)), smp.Symbol('Cl0*alpha')), (S(alpha(t)), smp.Symbol('S0')), 
                    (Ixx(t), smp.Symbol('Ixx')), (Iyy(t), smp.Symbol('Iyy')), (Izz(t), smp.Symbol('Izz')), 
                    (dIxx(t), smp.Symbol('dIxx')), (dIyy(t), smp.Symbol('dIyy')), (dIzz(t), smp.Symbol('dIzz')),
                    (T(t), smp.Symbol('T')), (m(t), smp.Symbol('m')), (dCoP, smp.Symbol('dCoP')), (dNoz, smp.Symbol('dNoz')),
                    (Tau0, smp.Symbol('Tau0')),  (f(t), smp.Symbol('f'))])


#Convert state to numpy array
Statenp = smp.matrix2numpy(Statesub)

#Display state equation vector, [dx, dy, dz, ddx, ddy, ddz, dphi, dtheta, dpsi, ddphi, ddtheta, ddpsi] 
print(State)
# print(alpha_exp)

Matrix([[\dot{x}(t)], [\dot{y}(t)], [\dot{z}(t)], [(-rho*((-(sin(phi(t))*sin(psi(t)) + sin(theta(t))*cos(phi(t))*cos(psi(t)))*\dot{y}(t) + (-sin(phi(t))*cos(psi(t)) + sin(psi(t))*sin(theta(t))*cos(phi(t)))*\dot{x}(t))*\dot{y}(t) - ((sin(phi(t))*sin(psi(t)) + sin(theta(t))*cos(phi(t))*cos(psi(t)))*\dot{z}(t) - \dot{x}(t)*cos(phi(t))*cos(theta(t)))*\dot{z}(t))*C_L(alpha(t))*S(alpha(t)) - (rho*sqrt(\dot{x}(t)**2 + \dot{y}(t)**2 + \dot{z}(t)**2)*C_D(alpha(t))*S(alpha(t))*\dot{x}(t) + 2*(((sin(phi(t))*sin(psi(t)) + sin(theta(t))*cos(phi(t))*cos(psi(t)))*cos(gamma_y(t)) + sin(gamma_y(t))*cos(psi(t))*cos(theta(t)))*cos(gamma_x(t)) - (sin(phi(t))*sin(theta(t))*cos(psi(t)) - sin(psi(t))*cos(phi(t)))*sin(gamma_x(t)))*(P(t) - 1)*T(t))*\sin_{\alpha}(t))/(2*\sin_{\alpha}(t)*m(t))], [(rho*(-((sin(phi(t))*sin(psi(t)) + sin(theta(t))*cos(phi(t))*cos(psi(t)))*\dot{y}(t) - (-sin(phi(t))*cos(psi(t)) + sin(psi(t))*sin(theta(t))*cos(phi(t)))*\dot{x}(t))*\dot{x}(t) + ((-sin(phi(t))*cos(psi(t)) + sin(psi(t))

In [6]:
State

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [7]:
print(Statenp)

[[dx]
 [dy]
 [dz]
 [(-Cl0*alpha*S0*rho*(dy*(dx*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi)) - dy*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))) - dz*(-dx*cos(phi)*cos(theta) + dz*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi)))) - (Cd0*S0*dx*rho*sqrt(dx**2 + dy**2 + dz**2) + 2*T*(P - 1)*(((sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))*cos(gy) + sin(gy)*cos(psi)*cos(theta))*cos(gx) - (sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))*sin(gx)))*\sin_{\alpha}(t))/(2*m*\sin_{\alpha}(t))]
 [(Cl0*alpha*S0*rho*(-dx*(-dx*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi)) + dy*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))) + dz*(-dy*cos(phi)*cos(theta) + dz*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi)))) - (Cd0*S0*dy*rho*sqrt(dx**2 + dy**2 + dz**2) + 2*T*(P - 1)*((-(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))*cos(gy) + sin(gy)*sin(psi)*cos(theta))*cos(gx) - (sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi))*sin(gx)))*\sin_{\alpha}(t))/(2*m*\sin_{\alpha}(t))]

In [32]:
#Parameters
T = 3000                    #Thrust
#2500 max, 1500 min
P_max = 0.4                 #Max throttle percentage
m0 = 285 #180                    #Mass     
m_dot0 = 1.75 #1.75 kg/s at 3kN
g = 9.8                     #Gravity
dNoz = 1.52 #0.7                  #Distance from CoM to Nozzle
t_final = 40                #Final simulation time
control_frequency = 15      #Hertz
dt = 1/control_frequency    #Control time step
l = 2.5                   
r = 0.2

#Parameters
Cd0 = 0.1 * 100
Cl0 = 0.01 * 100
S0 = 0.01
dCoP = 1
rho = 1.225

def Ixx(m):
    return (1/12) * m * (2.5 ** 2)
def Iyy(m):
    return (1/12) * m * (2.5 ** 2)
def Izz(m):
    return (1/2) * m * (0.2 ** 2)

def dIxx(dm):
    return (1/12) * dm * (2.5 ** 2)
def dIyy(dm):
    return (1/12) * dm * (2.5 ** 2)
def dIzz(dm):
    return (1/2) * dm * (0.2 ** 2)

Tau0 = 3.23

## Control Matrices

$$
A =
\begin{bmatrix}
0 & 0 & 1 & 0 & 0 & 0 \\[4pt]
0 & 0 & 0 & 1 & 0 & 0 \\[4pt]
0 & 0 & 0 & 0 & -g & 0 \\[4pt]
0 & 0 & 0 & 0 & 0 & 0 \\[10pt]
0 & 0 & 0 & 0 & 0 & 1 \\[4pt]
0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}

\\

B =
\begin{bmatrix}
0 & 0 \\[4pt]
0 & 0 \\[4pt]
 -g & 0 \\[4pt]
0 & -\dfrac{T}{m} \\[4pt]
0 & 0 \\[4pt]
\dfrac{d_{Noz} m g}{I_{yy}} & 0
\end{bmatrix}
\\
\begin{bmatrix}
    \dot{x} \\
    \dot{z} \\
    \ddot{x} \\
    \ddot{z} \\
    \dot{\theta} \\
    \ddot{\theta}
\end{bmatrix} 
\approx
A \begin{bmatrix}
    {x} \\
    {z} \\
    \dot{x} \\
    \dot{z} \\
    {\theta} \\
    \dot{\theta}
\end{bmatrix} 
+
B \begin{bmatrix}
    \gamma \\
    P-P_{hover} \\
\end{bmatrix} 
\\

\begin{bmatrix}
    \gamma \\
    P-P_{hover} \\
\end{bmatrix} 
= - K \begin{bmatrix}
    {x-0} \\
    {z-z_{ref}} \\
    \dot{x}-0 \\
    \dot{z}-\dot{z}_{ref} \\
    {\theta}-0 \\
    \dot{\theta}-0
\end{bmatrix} 
$$

In [12]:
#Control matrices

#Continuous control matrices
Atc = np.zeros([6,6])
Atc[0,2] = 1
Atc[1,3] = 1
Atc[2,4] = -g
Atc[4,5] = 1

Btc = np.zeros([6,2])
Btc[2,0] = -g
Btc[3,1] = -T/m0
Btc[5,0] = dNoz*m0*g/Iyy(m0)

#Discrete control matrices
Concatenated = np.concatenate((np.concatenate((Atc, Btc), axis = 1), np.zeros((2,8))),axis=0)
Exp_Concat = np.round(sp.linalg.expm(Concatenated * dt), decimals=4)
Atd = Exp_Concat[:6,:6]
Btd = Exp_Concat[:6,6:]

In [13]:
print(f'Atc \n {Atc}\n Btc \n{Btc}\n Atd \n {Atd}\n Btd \n {Btd}')

Atc 
 [[ 0.   0.   1.   0.   0.   0. ]
 [ 0.   0.   0.   1.   0.   0. ]
 [ 0.   0.   0.   0.  -9.8  0. ]
 [ 0.   0.   0.   0.   0.   0. ]
 [ 0.   0.   0.   0.   0.   1. ]
 [ 0.   0.   0.   0.   0.   0. ]]
 Btc 
[[  0.           0.        ]
 [  0.           0.        ]
 [ -9.8          0.        ]
 [  0.         -10.52631579]
 [  0.           0.        ]
 [ 28.60032      0.        ]]
 Atd 
 [[ 1.000e+00  0.000e+00  6.670e-02  0.000e+00 -2.180e-02 -5.000e-04]
 [ 0.000e+00  1.000e+00  0.000e+00  6.670e-02  0.000e+00  0.000e+00]
 [ 0.000e+00  0.000e+00  1.000e+00  0.000e+00 -6.533e-01 -2.180e-02]
 [ 0.000e+00  0.000e+00  0.000e+00  1.000e+00  0.000e+00  0.000e+00]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.000e+00  6.670e-02]
 [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00  1.000e+00]]
 Btd 
 [[-0.022   0.    ]
 [ 0.     -0.0234]
 [-0.6672  0.    ]
 [ 0.     -0.7018]
 [ 0.0636  0.    ]
 [ 1.9067  0.    ]]
